# Memory Tax: public data loader and net-value receipt

This notebook accompanies [He et al., arXiv:2610.34073](https://arxiv.org/abs/2610.34073). It loads two public PersonaMem-v2 examples and implements the paper's query-level comparison of an agent memory system against naive RAG. No API key is required.

In [ ]:
import csv
import io
import json
import urllib.request
from pprint import pprint

## 1. Load a public benchmark used by the paper

The URL is pinned to the same PersonaMem-v2 revision as the paper's official downloader. We fetch only the benchmark CSV, not the much larger chat histories.

In [ ]:
REVISION = 'ed956dea41521fc4499acbc63f966e0fd3c053ba'
DATA_URL = (
    'https://huggingface.co/datasets/bowen-upenn/PersonaMem-v2/resolve/'
    f'{REVISION}/benchmark/text/benchmark.csv'
)

with urllib.request.urlopen(DATA_URL) as response:
    text = response.read().decode('utf-8-sig')
rows = list(csv.DictReader(io.StringIO(text)))

print(f'Loaded {len(rows)} PersonaMem-v2 benchmark rows.')
print('Columns:', list(rows[0]))
print('\nFirst two compact examples:')
example_fields = ['persona_id', 'user_query', 'correct_answer', 'incorrect_answers', 'preference']
for row in rows[:2]:
    example = {field: row[field] for field in example_fields}
    print(json.dumps(example, indent=2, ensure_ascii=False))

## 2. A tiny paired evaluation

Each row represents the *same query* answered by naive RAG and by a memory system. These illustrative correctness labels are synthetic; replace them with judged outputs from your own experiment.

In [ ]:
paired_results = [
    {'query_id': 'q1', 'rag_correct': True,  'memory_correct': True},
    {'query_id': 'q2', 'rag_correct': False, 'memory_correct': True},   # benefit
    {'query_id': 'q3', 'rag_correct': True,  'memory_correct': False},  # tax
    {'query_id': 'q4', 'rag_correct': True,  'memory_correct': False},  # tax
    {'query_id': 'q5', 'rag_correct': False, 'memory_correct': False},
    {'query_id': 'q6', 'rag_correct': True,  'memory_correct': True},
]
pprint(paired_results)

## 3. Produce the memory-tax receipt

Benefit and tax expose *which system uniquely succeeded*. Their difference is net value. Dividing net value by the number of queries gives the accuracy change relative to RAG.

In [ ]:
def memory_tax_receipt(records):
    groups = {
        'benefit': [],
        'tax': [],
        'both_correct': [],
        'both_wrong': [],
    }
    for row in records:
        rag, memory = row['rag_correct'], row['memory_correct']
        if memory and not rag:
            group = 'benefit'
        elif rag and not memory:
            group = 'tax'
        elif rag and memory:
            group = 'both_correct'
        else:
            group = 'both_wrong'
        groups[group].append(row['query_id'])

    n = len(records)
    benefit = len(groups['benefit'])
    tax = len(groups['tax'])
    net_value = benefit - tax
    rag_accuracy = sum(row['rag_correct'] for row in records) / n
    memory_accuracy = sum(row['memory_correct'] for row in records) / n
    return {
        'queries': n,
        'groups': groups,
        'benefit_count': benefit,
        'tax_count': tax,
        'net_value': net_value,
        'normalized_net_value': net_value / n,
        'rag_accuracy': rag_accuracy,
        'memory_accuracy': memory_accuracy,
        'accuracy_change': memory_accuracy - rag_accuracy,
    }

receipt = memory_tax_receipt(paired_results)
pprint(receipt)
assert abs(receipt['normalized_net_value'] - receipt['accuracy_change']) < 1e-12

### Interpretation

In this toy run, memory creates one unique success but destroys two RAG successes, so its net value is −1 query. The final assertion verifies the accounting identity. A full reproduction must generate and judge paired answers on the paper's pinned benchmark subsets; this notebook only isolates the central diagnostic.